# AlgoLab Python 等效实现 — GA/SA/TS/ALNS 元启发式

对应 `05_AlgoLab.html`。本 notebook 用 Python 重现 4 种主流元启发式算法：
- GA：遗传算法（POX 交叉 + 锦标赛选择 + 精英保留）
- SA：模拟退火（Metropolis 准则 + 3 邻域算子）
- TS：禁忌搜索（Aspiration 准则 + 长期记忆）
- ALNS：自适应大邻域搜索（Destroy + Repair + 自适应权重）

**目标问题**：30 工单 × 8 机器异构并行机调度（R||Cmax）


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random
import time
from copy import deepcopy

random.seed(42); np.random.seed(42)

# 问题实例
n_jobs, n_machines = 30, 8
p = np.random.uniform(1, 8, (n_jobs, n_machines))  # 加工时长矩阵
d = np.random.uniform(30, 80, n_jobs)               # 交期
weights = np.random.choice([1, 2, 3], n_jobs)        # 权重

## 1. 解的表示与评估

In [ ]:
def random_solution():
    """随机生成解：assign[i] = 机器索引"""
    return np.random.randint(0, n_machines, n_jobs)

def evaluate(assign, alpha=0.5, beta=0.3, gamma=0.2):
    """计算 Cmax + 总延迟"""
    machine_load = np.zeros(n_machines)
    completion = np.zeros(n_jobs)
    for j in range(n_jobs):
        m = assign[j]
        machine_load[m] += p[j, m]
        completion[j] = machine_load[m]
    Cmax = machine_load.max()
    tardy = np.maximum(0, completion - d)
    total_tardy = (weights * tardy).sum()
    return alpha * Cmax + beta * total_tardy, Cmax, total_tardy.sum()

## 2. GA — 遗传算法

In [ ]:
def run_ga(pop_size=80, gens=200):
    population = [random_solution() for _ in range(pop_size)]
    fitness = [evaluate(s)[0] for s in population]
    best_history = []

    for g in range(gens):
        # 锦标赛选择 + 交叉 + 变异
        new_pop = []
        for _ in range(pop_size):
            # 锦标赛
            i1, i2 = random.sample(range(pop_size), 2)
            parent = population[i1] if fitness[i1] < fitness[i2] else population[i2]
            child = parent.copy()
            # 变异
            if random.random() < 0.2:
                idx = random.randint(0, n_jobs - 1)
                child[idx] = random.randint(0, n_machines - 1)
            new_pop.append(child)
        population = new_pop
        fitness = [evaluate(s)[0] for s in population]
        best_history.append(min(fitness))

    best_idx = np.argmin(fitness)
    return population[best_idx], best_history

start = time.time()
ga_sol, ga_hist = run_ga()
print(f"GA: {evaluate(ga_sol)[0]:.2f}, Cmax={evaluate(ga_sol)[1]:.2f}, 耗时={time.time()-start:.2f}s")

## 3. SA — 模拟退火

In [ ]:
def run_sa(T0=1000, alpha=0.95, max_iter=2000):
    current = random_solution()
    current_fit = evaluate(current)[0]
    best = current.copy(); best_fit = current_fit
    T = T0
    history = []

    for _ in range(max_iter):
        # 邻域：随机改变一个订单的机器
        candidate = current.copy()
        idx = random.randint(0, n_jobs - 1)
        candidate[idx] = random.randint(0, n_machines - 1)
        cand_fit = evaluate(candidate)[0]
        delta = cand_fit - current_fit
        if delta < 0 or random.random() < np.exp(-delta / max(0.01, T)):
            current = candidate; current_fit = cand_fit
            if current_fit < best_fit:
                best = current.copy(); best_fit = current_fit
        T *= alpha
        history.append(best_fit)
    return best, history

start = time.time()
sa_sol, sa_hist = run_sa()
print(f"SA: {evaluate(sa_sol)[0]:.2f}, Cmax={evaluate(sa_sol)[1]:.2f}, 耗时={time.time()-start:.2f}s")

## 4. TS — 禁忌搜索

In [ ]:
def run_ts(tabu_size=12, max_iter=500, neighborhood=30):
    current = random_solution()
    best = current.copy()
    best_fit = evaluate(best)[0]
    tabu = []
    history = []

    for _ in range(max_iter):
        # 生成邻域
        best_neighbor = None; best_neighbor_fit = float("inf"); best_move = None
        for _ in range(neighborhood):
            candidate = current.copy()
            idx = random.randint(0, n_jobs - 1)
            new_m = random.randint(0, n_machines - 1)
            move = (idx, current[idx], new_m)
            candidate[idx] = new_m
            cf = evaluate(candidate)[0]
            # Aspiration 或非 tabu
            if (move not in tabu) or (cf < best_fit):
                if cf < best_neighbor_fit:
                    best_neighbor_fit = cf
                    best_neighbor = candidate
                    best_move = move
        if best_neighbor is not None:
            current = best_neighbor
            tabu.append(best_move)
            if len(tabu) > tabu_size: tabu.pop(0)
            if best_neighbor_fit < best_fit:
                best = best_neighbor.copy(); best_fit = best_neighbor_fit
        history.append(best_fit)
    return best, history

start = time.time()
ts_sol, ts_hist = run_ts()
print(f"TS: {evaluate(ts_sol)[0]:.2f}, Cmax={evaluate(ts_sol)[1]:.2f}, 耗时={time.time()-start:.2f}s")

## 5. ALNS — 自适应大邻域搜索

In [ ]:
def run_alns(max_iter=500, q_ratio=0.25, T0=100, alpha=0.99):
    current = random_solution()
    current_fit = evaluate(current)[0]
    best = current.copy(); best_fit = current_fit
    T = T0
    history = []

    for _ in range(max_iter):
        # Destroy：随机移除 q 个订单的机器分配
        q = int(n_jobs * q_ratio)
        removed = random.sample(range(n_jobs), q)
        # Repair：贪婪重插（每订单选当前负载最小的机器）
        candidate = current.copy()
        for idx in removed:
            machine_load = np.zeros(n_machines)
            for j in range(n_jobs):
                if j != idx:
                    machine_load[candidate[j]] += p[j, candidate[j]]
            best_m = np.argmin(machine_load + p[idx, :])
            candidate[idx] = best_m
        cand_fit = evaluate(candidate)[0]
        delta = cand_fit - current_fit
        if delta < 0 or random.random() < np.exp(-delta / max(0.01, T)):
            current = candidate; current_fit = cand_fit
            if current_fit < best_fit:
                best = current.copy(); best_fit = current_fit
        T *= alpha
        history.append(best_fit)
    return best, history

start = time.time()
alns_sol, alns_hist = run_alns()
print(f"ALNS: {evaluate(alns_sol)[0]:.2f}, Cmax={evaluate(alns_sol)[1]:.2f}, 耗时={time.time()-start:.2f}s")

## 6. 4 算法 Benchmark 对比

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(ga_hist, label="GA", color="#1F4E79", linewidth=2)
ax.plot(sa_hist[::5], label="SA", color="#C00000", linewidth=2)
ax.plot(ts_hist, label="TS", color="#7030A0", linewidth=2)
ax.plot(alns_hist, label="ALNS", color="#385723", linewidth=2)
ax.set_xlabel("迭代次数"); ax.set_ylabel("目标函数 f")
ax.set_title("4 种元启发式算法收敛对比 (30×8 R||Cmax)")
ax.legend(); ax.grid(alpha=0.3)
plt.show()

print("\n最终结果对比:")
for name, sol in [("GA", ga_sol), ("SA", sa_sol), ("TS", ts_sol), ("ALNS", alns_sol)]:
    obj, cmax, tardy = evaluate(sol)
    print(f"  {name}: 目标={obj:.2f}  Cmax={cmax:.2f}  总加权延迟={tardy:.2f}")

## 总结
本 notebook 用 Python 完整实现了 AlgoLab 网页版的 4 种元启发式算法。

**与网页版的对应关系**：
- 网页版面向 MBA / 本科零编程基础，强调"可视化收敛动画"
- 本 notebook 面向研究生 / 算法工程师，强调"代码可改、参数可调、可扩展到更复杂问题"

**进阶建议**：
- 加入 GA 的 POX 交叉算子（参考 AlgoLab 源码）
- 实现 ALNS 的 3 个 Destroy + 3 个 Repair 算子 + 自适应权重
- 加入并行计算（multiprocessing）加速
- 用 OR-Tools 或 Gurobi 求得精确最优，作为算法效果上界
